<a href="https://colab.research.google.com/github/lili-cloud/ENSAI-2A-cinfo-TP4/blob/tp4_base/Copie_de_practical1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep learning - Practical session 1.2

Using what we learned in the first part of the practical session, we are going to
* preprocess the data,
* specify how to access the data,
* build a neural network, and
* train and evaluate your network.

We will work on the [Forest covertypes](https://archive.ics.uci.edu/dataset/31/covertype) dataset.
Let's load (or download if the first time) the dataset.

In [ ]:
!pip install scikit-learn

from sklearn.datasets import fetch_covtype

X, y = fetch_covtype(data_home='data', return_X_y=True)

## Exercise 1
1. Determine
    - the number of examples,
    - the input dimension (the number of attributes of input data), and
    - the set of the class labels.
2. Are the classes balanced?

Hint: You can use the [numpy.unique](https://numpy.org/doc/stable/reference/generated/numpy.unique.html) function with `return_counts=True`.

In [ ]:
import numpy as np
# 1. Nombre d'exemples
num_examples = X.shape[0]

# 2. Dimension d'entrée (nombre de caractéristiques ou attributs)
input_dim = X.shape[1]

# 3. Jeu d'étiquettes de classes et leur distribution
unique_classes, class_counts = np.unique(y, return_counts=True)

# 4. Vérification de l'équilibre des classes
classes_balanced = np.all(class_counts == class_counts[0])  # Vérifier si toutes les classes ont le même nombre d'exemples

# Affichage des résultats
print(f"Nombre d'exemples: {num_examples}")
print(f"Dimension d'entrée: {input_dim}")
print(f"Étiquettes de classes: {unique_classes}")
print(f"Distribution des classes: {class_counts}")
print(f"Les classes sont-elles équilibrées ? {'Oui' if classes_balanced else 'Non'}")


Nombre d'exemples: 581012
Dimension d'entrée: 54
Étiquettes de classes: [1 2 3 4 5 6 7]
Distribution des classes: [211840 283301  35754   2747   9493  17367  20510]
Les classes sont-elles équilibrées ? Non


## Exercise 2
Make any necessary changes to the `numpy.ndarray` of the class labels to obtain the following representation.

- Represent the classes with the first $K$ non-negative integers (starting from 0), that is $\{0, \ldots, K-1\}$, where $K$ is the number of classes.

- The `dtype` must be `int64` to work with PyTorch later [1].

Hint: use `numpy.astype`.
`numpy.int64`, `torch.int64`, and `torch.long` are all valid for specifying `int64`.

---

[1] Why does the indices tensor have to be Long dtype? PyTorch Forums, https://discuss.pytorch.org/t/why-does-the-indices-tensor-have-to-be-long-dtype/139675.

## Exercise 3
Split the dataset into
- a training set (100,000 examples),
- a validation set (100,000 examples), and
- a test set (the remainder).

Hint: You can use the [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function. Make sure to have (approximately) the same class distribution in all three sets by passing the class labels to the `stratify` parameter.

In [ ]:
# Train-validation-test split

# Assume we have X, y holding the whole dataset
# we want to split this dataset to 3 parts
from sklearn.model_selection import train_test_split

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=200_000, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=100_000, random_state=43, stratify=y_rest)



## Exercise 4
Convert the NumPy's arrays to Tensors. Remember that the input data must have `dtype` of `float32` in PyTorch.

Hint: use `torch_from_numpy` and its `to` method.

## Exercise 5
Create your `CustomDataset` class.

Reminder: it must inherit from the `torch.utils.data.Dataset` class and implement three methods:
* `__init__`: it receives and keeps all the necessary information for the other methods (e.g., the path of the data, the labels, transforms, etc.)
* `__len_`: it returns the number of examples in our dataset.
* `__getitem__`: loads and returns a sample from the dataset at the given index `idx`.

In [ ]:
from torch.utils.data import DataSet

class CustomDataset(Dataset):
  def__init__(self,  X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.y.size

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]



## Exercise 6
Create an instance `CustomDataset` and `DataLoader` for each of the training, validation, and test sets, with `batch_size=64` and `shuffle=True`.

In [ ]:
from torch.utils.data import DataLoader

train_dataset = CustomDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)


## Exercise 7
Create a class for a Multi-Layer Perceptron (MLP) with the following sequential architecture:
* First block: Linear layer (128 output features) + ReLU activation function
* Second block: Linear layer (64 output features) + ReLU activation function
* Final layer: Linear layer (7 output features)

Reminder: you have to create a class inheriting from the `torch.nn.Module` class and implementing two methods: `__init__` and `forward`.

Hint: See the documentation of [torch.nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html), [torch.nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) and [torch.nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html).

## Exercise 8
Define two functions `train_1epoch` and `evaluate`:
* `train_1epoch` will fit the model on the training data.
* `evaluate` will evaluate the model on either the validation set or test set.

In [ ]:
def train_1epoch(model, optimizer, device, dataloader):



## Exercise 9
Train the model that you defined in Exercise 7 on the training set. Use
* `1e-3` for the learning rate (step size),
* Stochastic Gradient Descent (SGD) for the optimizer, and
* the cross entropy loss for the loss function.

Also, print the accuracy and the loss on the validation set at the end of each epoch.

In [ ]:
learning_rate = 1e-3
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss()

## Exercise 10 (Advanced)
Is accuracy a good metric for this dataset? Which metric would be more relevant? How can we change the training procedure? Write a new version of the code according to your ideas.

Hint: The classes are highly imbalanced.

Furthermore, try changing the optimizer to [torch.optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html).

**Solution**: The classes are very imbalanced. In this case, accuracy can be a poor evaluation metric: the model can simply ignore the minority classes to achieve high accuracy. Balanced accuracy may be a better metric if we want to take minority classes into account.

For training, we can perform *cost-sensitive learning* by giving larger weights to samples from the minority classes in the cost function.

## Exercise 11
Change hyperparameters of your model (e.g., the architecture, the batch size, the optimizer, the loss function) as you want and train this new model. **Keep your previous models** and create a new model object because we want to compare them later.

## Exercise 12
When you are done experimenting with the hyperparameters of the model, you can finally evaluate the performance of your model on the test set. Choose your best model on the validation set and evaluate it on the test set. Compare its performance with the other models.